In [1]:
# input
clean_pred = "./tmp/clean_maxsep.csv"
fasta_file = "../filter/tmp/no_anno.fasta"
pred_info_file = "../filter/data/filtered.tsv"
cluster_file = "../filter/data/entryId-seqRepId-structRepId-unknownLevel.tsv"
# output
selected_file = "./data/dark_enzyme.tsv"

In [2]:
import pandas as pd

from Bio import SeqIO
id2seq = dict()
for r in SeqIO.parse(fasta_file, "fasta"):
    id2seq[r.id.split("-")[1]] = r.description


df_pred = pd.read_table(pred_info_file)
df_cluster = pd.read_table(cluster_file, header=None, names=["seq_id", "seq_rep_id", "struct_rep_id", "unknown_level"])
df_pred = pd.merge(df_pred, df_cluster, on="seq_id")
df_pred['name'] = df_pred['seq_id'].map(lambda x: id2seq[x].removeprefix(f"AFDB:AF-{x}-F1 "))
len(df_pred)

980

### clean pred

In [3]:
def parse_clean_output(file: str):

    records = []
    lines = []
    with open(file, "r") as f:
        lines = f.readlines()
    
    for l in lines:
        parts = l.split(",")
        seq_id = parts[0]

        ec_prob = []
        for p in parts[1:]:
            ec, prob = p.split("/")
            ec_prob.append((ec, float(prob)))
        ec_prob.sort(key=lambda x: x[1], reverse=True)

        ecs = [i[0] for i in ec_prob]
        probs = [i[1] for i in ec_prob]

        records.append({
            "seq_id": seq_id,
            "ec": ecs,
            "prob": probs
        })
    
    return pd.DataFrame(records)

In [4]:
df = parse_clean_output(clean_pred)
df['max_prob'] = df['prob'].map(lambda x: x[0])
df["seq_id"] = df['seq_id'].map(lambda x: x.split("-")[1])
df.sort_values(by=['max_prob'], ascending=False, inplace=True)
len(df)
df = df[df['max_prob'] > 0.2]
len(df)

980

29

### cluster info: each struct cluster gives one protein

In [5]:
df = pd.merge(df, df_pred, on='seq_id')
df['seq_id_length'] = df['seq_id'].str.len()
df = df.loc[
    df.sort_values(['max_prob', 'seq_id_length'], ascending=[False, True])
      .groupby('struct_rep_id')
      .head(1).index
]
df = df.drop(columns=['seq_id_length'])
len(df)
df

6

,seq_id,ec,prob,max_prob,pred_seq_num,proba,metal_type,metal_group_type,avg_plddt,plddt,site,len,seq_rep_id,struct_rep_id,unknown_level,name
0,A0A6C9M6E0,[EC:3.1.21.4],[0.7598],0.7598,"237,240,257,260,272,275,295,298","0.9668,0.9689,0.9604,0.9487,0.9709,0.9568,0.98...","0,0,0,4,0,0,0,0","1,1,1,1,1,1,1,1",89.738,"93.75,92.04,94.82,92.04,88.83,87.55,87.12,81.78","236,239,256,259;271,274,294,297",305,A0A090IEJ0,A0A1C0VLQ1,"1,2",Uncharacterized protein UA=A0A6C9M6E0 UI=A0A6C...
2,A0A483YIA5,[EC:3.1.21.4],[0.7223],0.7223,"46,55,79","0.8786,0.9731,0.8322","0,0,0","1,1,1",91.877,"96.88,94.44,94.71","45,54,78",336,A0A0H3GYE3,A0A7Z0MFA1,"1,2",Uncharacterized protein UA=A0A483YIA5 UI=A0A48...
4,A0A0A8RRG4,[EC:3.1.21.4],[0.6272],0.6272,"66,69,92,95","0.9634,0.9103,0.9404,0.8697","0,0,0,0","1,1,1,1",89.371,"92.14,91.71,90.42,91.28","65,68,91,94",440,A0A0X8BTU2,A0A0H4QGX8,2,Uncharacterized protein UA=A0A0A8RRG4 UI=A0A0A...
5,A0A486DF43,[EC:3.1.21.4],[0.6163],0.6163,"207,210,217,219","0.9926,0.9462,0.9649,0.8395","0,0,0,0","1,1,1,1",85.402,"86.23,86.23,88.97,90.11","206,209,216,218",256,A0A8B3RCH6,A0A3M8ESG9,1,Uncharacterized protein UA=A0A486DF43 UI=A0A48...
25,A0A483U4X0,[EC:2.1.1.86],[0.2705],0.2705,"55,58,69,72","0.9849,0.9867,0.9249,0.954","0,0,0,0","1,1,1,1",74.445,"90.09,92.94,91.43,92.1","54,57,68,71",122,V0AFL7,A0A449IRS4,1,Zinc ribbon domain-containing protein UA=A0A48...
27,F4NPC0,[EC:3.1.21.4],[0.2174],0.2174,"231,234,251,254","0.9915,0.9867,0.971,0.9588","0,0,0,0","1,1,1,1",87.839,"79.99,85.61,86.16,88.7","230,233,250,253",260,A0A5Q2K4R6,A0A151KV72,2,Uncharacterized protein UA=F4NPC0 UI=F4NPC0_EC...


### protein name

In [ ]:
# remove A0A483U4X0 (zinc-ribbon)
df = df[df['seq_id'] != "A0A483U4X0"]
df.to_csv(selected_file, sep="\t", index=None)

# note that the old version gave:

# #A0A193LMF5
# #F4TN75
# A0A486DF43
# #A0A376SG87
# F4NPC0
# #A0A8B0UYQ9
# A0A6C9M6E0
# A0A0A8RRG4
# A0A483YIA5